# HTSPF Vision Benchmark — CIFAR-100
**Runs the full CIFAR-100 benchmark grid on Kaggle GPU.**
- 5 HTSPF ablations × 5 seeds = 25 runs
- 3 vision baselines (ResNet-18, ViT-Small, Perceiver-IO) × 5 seeds = 15 runs
- **Total: 40 runs × 50 epochs each**
- Est. wall time on T4: ~4-6 hours

## Setup
1. Upload your repo as a Kaggle Dataset (zip the HTSPF folder, upload via Kaggle Datasets UI)
2. Attach it to this notebook as `/kaggle/input/htspf-repo`
3. Enable GPU accelerator (T4 x 2 recommended)
4. Run all cells

In [ ]:
# Install dependencies
!pip install -q torch torchvision datasets pyyaml ptflops scipy

In [ ]:
import os, shutil

REPO_SRC = '/kaggle/input/htspf-repo/HTSPF'
REPO_DST = '/kaggle/working/HTSPF'

if os.path.exists(REPO_DST):
    shutil.rmtree(REPO_DST)
shutil.copytree(REPO_SRC, REPO_DST)
os.chdir(REPO_DST)
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
import subprocess, sys

CONFIG  = 'configs/experiment.yaml'
OUT_DIR = 'results/raw'
SEEDS   = [0, 1, 2, 3, 4]

ABLATIONS        = ['HTSPF_Full', 'HTSPF_noHCAA', 'HTSPF_noASG', 'HTSPF_noLDWT', 'HTSPF_noConflict']
VISION_BASELINES = ['resnet18', 'vit_small', 'perceiver_io']

os.makedirs(OUT_DIR, exist_ok=True)

def run_experiment(model, dataset, seed):
    label = f'{model} | {dataset} | seed={seed}'
    print(f'\n>>> {label}')
    result = subprocess.run(
        [sys.executable, 'src/train.py',
         '--model',   model,
         '--dataset', dataset,
         '--seed',    str(seed),
         '--config',  CONFIG,
         '--output',  OUT_DIR],
        capture_output=False, text=True
    )
    if result.returncode != 0:
        print(f'ERROR in {label} (returncode={result.returncode})')
    return result.returncode

errors = []
print('='*60)
print('HTSPF VISION BENCHMARK - CIFAR-100')
print(f'{len(ABLATIONS)+len(VISION_BASELINES)} models x {len(SEEDS)} seeds = {(len(ABLATIONS)+len(VISION_BASELINES))*len(SEEDS)} runs')
print('='*60)

In [ ]:
# HTSPF Ablations on CIFAR-100
print('=== HTSPF Ablations ===')
for model in ABLATIONS:
    for seed in SEEDS:
        rc = run_experiment(model, 'cifar100', seed)
        if rc != 0:
            errors.append(f'{model}_cifar100_seed{seed}')
print(f'Ablations done. Errors: {errors if errors else "none"}')

In [ ]:
# Vision Baselines on CIFAR-100
print('=== Vision Baselines ===')
for model in VISION_BASELINES:
    for seed in SEEDS:
        rc = run_experiment(model, 'cifar100', seed)
        if rc != 0:
            errors.append(f'{model}_cifar100_seed{seed}')
print(f'Baselines done. Errors: {errors if errors else "none"}')

In [ ]:
# Verify outputs and zip for download
import json, glob

json_files = sorted(glob.glob(f'{OUT_DIR}/*.json'))
print(f'Total result files: {len(json_files)} (expected 40)')
for f in json_files:
    with open(f) as fh:
        d = json.load(fh)
    print(f'  {os.path.basename(f):50s}  acc={d.get("test_acc", "?"):.2f}%')

shutil.make_archive('/kaggle/working/cifar100_results', 'zip', OUT_DIR)
print('\nDownload cifar100_results.zip from Kaggle output panel.')